# 🍽️ Cantina Escolar — Apache Spark

Notebook independente da seção Spark do projeto original.

**Etapas:** carregar dados Silver com PySpark; preparar e cruzar vendas, cardápio, alunos e clima; executar Spark SQL; conferir o faturamento; simular vendas chegando dia a dia com Structured Streaming.

> Ajuste `PASTA_DADOS`. O leitor aceita `.csv` e `.parquet`. Se os arquivos estiverem em `.xlsx`, exporte-os como CSV antes de executar.

## 0. Instalação e inicialização

In [ ]:
%pip -q install pyspark

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("CantinaEscolar_Spark").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Versão:", spark.version)

## 1. Configuração e leitura

Arquivos necessários:
- `vendas`: id_venda, data_hora, id_aluno, id_produto, quantidade, valor_total, forma_pagamento
- `cardapio`: id_produto, nome_produto, categoria, custo_unitario
- `alunos`: id_aluno, ano_letivo, turma
- `clima`: data, temperatura_media, temperatura_max, condicao, precipitacao_mm

In [ ]:
# Google Colab: monte o Drive antes, se necessário.
# Exemplo: /content/drive/MyDrive/P.I-Data-Science/dados/silver
PASTA_DADOS = "/content/P.I-Data-Science/dados/silver"

def ler_tabela(nome):
    parquet = os.path.join(PASTA_DADOS, nome + ".parquet")
    csv = os.path.join(PASTA_DADOS, nome + ".csv")
    if os.path.exists(parquet):
        return spark.read.parquet(parquet)
    if os.path.exists(csv):
        return (spark.read.option("header", True)
                .option("inferSchema", True)
                .option("encoding", "UTF-8").csv(csv))
    raise FileNotFoundError(f"Não encontrei {nome}.parquet nem {nome}.csv em {PASTA_DADOS}")

vendas = ler_tabela("vendas")
cardapio = ler_tabela("cardapio")
alunos = ler_tabela("alunos")
clima = ler_tabela("clima")

for nome, tabela in {"vendas": vendas, "cardapio": cardapio,
                     "alunos": alunos, "clima": clima}.items():
    print(f"{nome}: {tabela.count():,} linhas")
    tabela.printSchema()

## 2. Preparação e enriquecimento das vendas

In [ ]:
vendas = (
    vendas.withColumn("data_hora", F.to_timestamp("data_hora"))
    .withColumn("data", F.to_date("data_hora"))
    .withColumn("ano_letivo", F.year("data_hora"))
    .withColumn("mes", F.date_format("data_hora", "yyyy-MM"))
    .withColumn("hora", F.hour("data_hora"))
    .withColumn("quantidade", F.col("quantidade").cast("int"))
    .withColumn("valor_total", F.col("valor_total").cast("double"))
)

cardapio = cardapio.dropDuplicates(["id_produto"])
alunos = (alunos.withColumn("id_aluno", F.col("id_aluno").cast("string"))
          .withColumn("ano_letivo", F.col("ano_letivo").cast("int")))
vendas = vendas.withColumn("id_aluno", F.col("id_aluno").cast("string"))
clima = clima.withColumn("data", F.to_date("data"))

# A junção dos alunos usa id_aluno + ano_letivo, pois IDs podem se repetir por ano.
ve = (
    vendas.join(cardapio.select("id_produto", "nome_produto", "categoria", "custo_unitario"),
                on="id_produto", how="left")
    .join(alunos.select("id_aluno", "ano_letivo", "turma"),
          on=["id_aluno", "ano_letivo"], how="left")
    .join(clima.select("data", "temperatura_media", "temperatura_max",
                       "condicao", "precipitacao_mm"), on="data", how="left")
    .withColumn("lucro", F.col("valor_total") -
                F.col("quantidade") * F.col("custo_unitario"))
    .withColumn("turma", F.coalesce("turma", F.lit("Não identificado")))
    .withColumn("categoria", F.coalesce("categoria", F.lit("Não identificado")))
    .withColumn("nome_produto", F.coalesce("nome_produto", F.lit("Não identificado")))
)

print("Vendas enriquecidas:", f"{ve.count():,}", "linhas")
ve.show(10, truncate=False)

## 3. Spark SQL — faturamento por categoria e mês

In [ ]:
ve.createOrReplaceTempView("vendas")

resultado_sql = spark.sql("""
    SELECT mes, categoria,
           SUM(quantidade) AS itens,
           ROUND(SUM(valor_total), 2) AS faturamento
    FROM vendas
    GROUP BY mes, categoria
    ORDER BY mes, faturamento DESC
""")
resultado_sql.show(30, truncate=False)

## 4. Conferência do faturamento total

In [ ]:
total_spark = ve.agg(F.sum("valor_total").alias("total")).first()["total"]
print(f"Faturamento total — Spark: R$ {total_spark:,.2f}")

## 5. Structured Streaming — simulação de vendas ao longo dos dias

Cada dia será gravado em um JSON separado. O Spark monitora a pasta de entrada e atualiza o faturamento por categoria quando novos arquivos chegam.

In [ ]:
N_DIAS_SIMULADOS = 10
PASTA_STREAM = "/content/stream_cantina"
PASTA_ENTRADA = os.path.join(PASTA_STREAM, "entrada")
PASTA_CHECKPOINT = os.path.join(PASTA_STREAM, "checkpoint")
PASTA_TEMP = os.path.join(PASTA_STREAM, "temporarios")

shutil.rmtree(PASTA_STREAM, ignore_errors=True)
os.makedirs(PASTA_ENTRADA, exist_ok=True)
os.makedirs(PASTA_CHECKPOINT, exist_ok=True)
os.makedirs(PASTA_TEMP, exist_ok=True)

esquema = StructType([
    StructField("data", StringType(), True),
    StructField("hora", IntegerType(), True),
    StructField("categoria", StringType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("valor_total", DoubleType(), True),
])

fluxo = spark.readStream.schema(esquema).json(PASTA_ENTRADA)
painel = (fluxo.groupBy("categoria")
          .agg(F.sum("quantidade").alias("itens"),
               F.round(F.sum("valor_total"), 2).alias("faturamento")))

consulta = (painel.writeStream.format("memory")
            .queryName("painel_ao_vivo")
            .outputMode("complete")
            .option("checkpointLocation", PASTA_CHECKPOINT)
            .start())
print("Streaming iniciado.")

In [ ]:
# Prepara os dados que serão enviados em lotes diários
base_stream = (
    ve.select("data", "hora", "categoria", "quantidade", "valor_total")
    .withColumn("data", F.date_format("data", "yyyy-MM-dd"))
    .withColumn("hora", F.col("hora").cast("int"))
    .withColumn("quantidade", F.col("quantidade").cast("int"))
    .withColumn("valor_total", F.col("valor_total").cast("double"))
)

dias = [r["data"] for r in (
    base_stream.select("data").distinct().orderBy("data")
    .limit(N_DIAS_SIMULADOS).collect()
)]

for numero, dia in enumerate(dias, start=1):
    lote = base_stream.filter(F.col("data") == dia)
    temporario = os.path.join(PASTA_TEMP, f"lote_{numero:03d}.json")
    destino = os.path.join(PASTA_ENTRADA, f"lote_{numero:03d}.json")

    # Escreve fora da pasta monitorada e move ao terminar, evitando leitura parcial.
    lote.toPandas().to_json(temporario, orient="records",
                            lines=True, force_ascii=False)
    os.replace(temporario, destino)
    consulta.processAllAvailable()

    acumulado = spark.sql(
        "SELECT SUM(itens) AS itens, SUM(faturamento) AS faturamento "
        "FROM painel_ao_vivo"
    ).first()

    print(f"Lote {numero:02d} ({dia}) → "
          f"{acumulado['itens'] or 0:,.0f} itens | "
          f"R$ {acumulado['faturamento'] or 0:,.2f}")

print("Painel final por categoria:")
spark.sql("SELECT * FROM painel_ao_vivo ORDER BY faturamento DESC").show(truncate=False)
consulta.stop()
print("Streaming encerrado.")

## 6. Encerramento

O notebook realizou a leitura com PySpark, os joins, a consulta Spark SQL e a simulação de Structured Streaming.

Execute a célula abaixo ao terminar.

In [ ]:
spark.stop()
print("Sessão Spark encerrada.")